Published on April 30, 2023. By Marília Prata, mpwolke.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt

import math

#Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
with open('data/raw/txt/romance/domCasmurro.txt', 'r', encoding='utf8') as f:
    data = f.read()

In [ ]:
print("Extract: ", data[:50])

In [ ]:
print("Length: ", len(data))

In [3]:
chars = list(set(data))

In [4]:
indexer = {char: index for (index, char) in enumerate(chars)}

In [5]:
indexed_data = []
for c in data:
    indexed_data.append(indexer[c])
    
print("Indexed extract: ", indexed_data[:50])
print("Length: ", len(indexed_data))

Indexed extract:  [86, 96, 17, 65, 12, 95, 27, 17, 38, 78, 78, 96, 19, 19, 12, 70, 54, 63, 36, 81, 71, 75, 65, 54, 42, 23, 92, 40, 23, 42, 75, 19, 86, 75, 65, 36, 63, 36, 81, 71, 75, 19, 81, 17, 95, 65, 1, 96, 72, 99]
Length:  371506


In [6]:
def index2onehot(batch):
    
    batch_flatten = batch.flatten()
    onehot_flat = np.zeros((batch.shape[0] * batch.shape[1], len(indexer)))
    onehot_flat[range(len(batch_flatten)), batch_flatten] = 1
    onehot = onehot_flat.reshape((batch.shape[0], batch.shape[1], -1))
    
    return onehot

In [7]:
import torch
from torch import nn
from torch import optim
import torch.nn.functional as F

In [8]:
class LSTM(nn.Module):
    def __init__(self, char_length, hidden_size, n_layers):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.lstm = nn.LSTM(char_length, hidden_size, n_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, char_length)
        
    def forward(self, x, states):
        out, states = self.lstm(x, states)
        out = out.contiguous().view(-1, self.hidden_size)
        out = self.output(out)
        
        return out, states
    
    def init_states(self, batch_size):
        hidden = next(self.parameters()).data.new(self.n_layers, batch_size, self.hidden_size).zero_()
        cell = next(self.parameters()).data.new(self.n_layers, batch_size, self.hidden_size).zero_()
        states = (hidden, cell)
        
        return states

In [9]:
n_seq = 100 ## Number of sequences per batch
seq_length =  50
n_batches = math.floor(len(indexed_data) / n_seq / seq_length)

total_length = n_seq * seq_length * n_batches
x = indexed_data[:total_length]
x = np.array(x).reshape((n_seq,-1))

In [10]:
model = LSTM(len(chars), 256, 2)
model

LSTM(
  (lstm): LSTM(101, 256, num_layers=2, batch_first=True)
  (output): Linear(in_features=256, out_features=101, bias=True)
)

#Optimizer

In [11]:
loss_function = nn.CrossEntropyLoss()
torch.autograd.set_detect_anomaly(True)

optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 2 # 20

In [12]:
losses = []

for e in range(1, epochs+1):
    states = model.init_states(n_seq)
    batch_loss = []
    
    for b in range(0, x.shape[1], seq_length):
        x_batch = x[:,b:b+seq_length]
        
        if b == x.shape[1] - seq_length:
            y_batch = x[:,b+1:b+seq_length]
            y_batch = np.hstack((y_batch, indexer["."] * np.ones((y_batch.shape[0],1))))
        else:
            y_batch = x[:,b+1:b+seq_length+1]
        
        x_onehot = torch.Tensor(index2onehot(x_batch))
        y = torch.Tensor(y_batch).view(n_seq * seq_length)
        
        pred, states = model(x_onehot, states)
        loss = loss_function(pred, y.long())
        optimizer.zero_grad()
        loss.backward(retain_graph=True)
        optimizer.step()
        
        batch_loss.append(loss.item())
        
    losses.append(np.mean(batch_loss))
    
    if e%1 == 0:
        print("epoch: ", e, "... Loss function: ", losses[-1])

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [1024, 256]] is at version 2; expected version 1 instead. Hint: the backtrace further above shows the operation that failed to compute its gradient. The variable in question was changed in there or anywhere later. Good luck!

#No Loss function below. and I couldn't fix above adding  torch.autograd.set_detect_anomaly(True)

So, why did they wish "Good Luck!" at the end (on the warning above)?

In [ ]:
x_range = range(len(losses))
plt.plot(x_range, losses)
plt.xlabel("epochs")
plt.ylabel("Loss function")
plt.show()

In [ ]:
starter = "Entrou a dizer de mim nomes feios, e acabou alcunhando-me Dom Casmurro."
states = None
for ch in starter:
    x = np.array([[indexer[ch]]])
    x = index2onehot(x)
    x = torch.Tensor(x)
    
    pred, states = model(x, states)

counter = 0
while starter[-1] != "." and counter < 50:
    counter += 1
    x = np.array([[indexer[starter[-1]]]])
    x = index2onehot(x)
    x = torch.Tensor(x)
    
    pred, states = model(x, states)
    pred = F.softmax(pred, dim=1)
    p, top = pred.topk(10)
    p = p.detach().numpy()[0]
    top = top.numpy()[0]
    index = np.random.choice(top, p=p/p.sum())
    
    starter += chars[index]
print(starter)

#That's it?  Dom Casmurro

"Neighbors, who don't like my reclusive and quiet habits, gave way to the nickname, which finally stuck. That's not why I was angry. I told the anecdote to friends in the city, and they, as a joke, called me that, some in notes: "Dom Casmurro, I'm going to have dinner with you on Sunday".

Author: Machado de Assis

#Predict if Capitu cheated on her husband (Bentinho) with his best friend, Escobar.

https://www.record.com.br/produto/dom-casmurro-edicao-de-bolso/

![](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRiud2m8UDwK-O5-PH4zQBjqi1-2Vi_BGaAmg&usqp=CAU)

#Acknowledgements:

Saman Siadati https://www.kaggle.com/code/samansiadati/pytorch-alice-in-wonderland